# CI/CD — End-to-End SageMaker Pipeline for the Pneumonia CNN
## AAI-540-02 Final Project · Group 4 · Standard Medical Models

This notebook is the **CI/CD deliverable** for the project. It authors a SageMaker Pipelines DAG that ties the work from `data_preparations.ipynb` and `CNN_Model.ipynb` into a single re-runnable, parameterizable, lineage-tracked workflow with a model-quality gate and a Model Registry hand-off.

```
         ┌────────────────────┐
         │ ProcessingStep:    │   (read preprocessed-images manifest;
         │ Split & Manifest   │    stratified 40/10/10/40 split;
         └─────────┬──────────┘    emit train/val/test/prod CSVs)
                   │
                   ▼
         ┌────────────────────┐
         │ TrainingStep:      │   (TensorFlow estimator on ml.m5.xlarge;
         │ Train CNN          │    early stopping; ONNX export)
         └─────────┬──────────┘
                   │
                   ▼
         ┌────────────────────┐
         │ ProcessingStep:    │   (precision / recall / F1 / AUC at
         │ Evaluate Model     │    threshold=0.4 → evaluation.json)
         └─────────┬──────────┘
                   │
                   ▼
         ┌────────────────────┐
         │ ConditionStep:     │
         │ F1 ≥ threshold ?   │
         └─────┬──────────┬───┘
          pass │          │ fail
               ▼          ▼
    Register + Create   FailStep
      + BatchTransform  (records the F1 in the error message)
```

**Prerequisite:** `data_preparations.ipynb` must have been run with `IS_DATA_OWNER=True` so that `s3://pneumonia-data-set-group-4/preprocessed-images/` is populated and `s3://pneumonia-data-set-group-4/pneumonia-project/metadata/image_metadata.csv` exists. This pipeline reads those artifacts as inputs; it does not regenerate them.

**Why a pipeline rather than running the notebooks end-to-end?** Once registered, the same DAG can be re-executed with different parameters (tighter F1 gate, GPU instance type, longer training) without touching any code. SageMaker tracks lineage, metrics, and approval status. The notebook is the *author* of the pipeline; SageMaker is the runtime.

## Step 1 · Setup

Pin `sagemaker` to the 2.x line (Pipelines APIs we use don't exist on 1.x) and import the SDK pieces. The key import is `PipelineSession`: when an estimator or processor's `.fit()` / `.run()` is called against a `PipelineSession`, it doesn't actually execute — it returns step args that the pipeline assembles into a DAG. This is how the notebook authors a pipeline without running anything yet.


In [ ]:
# Pin sagemaker to 2.x — Pipelines APIs we use don't exist on 1.x, and
# SageMaker Studio's base image sometimes ships a stale version.
%pip uninstall sagemaker -y -q
%pip install "sagemaker>=2.0,<3.0" pyathena awswrangler "boto3>1.17.21" -q

In [ ]:
import json
import os

import boto3
import sagemaker
from sagemaker.workflow.pipeline_context import PipelineSession

from config import BUCKET_NAME

# Two sessions: one for synchronous calls (uploads, downloads, lookups),
# one that defers .run() / .fit() into pipeline step args.
sagemaker_session = sagemaker.session.Session()
pipeline_session = PipelineSession()

region = sagemaker_session.boto_region_name
role = sagemaker.get_execution_role()
default_bucket = sagemaker_session.default_bucket()
bucket = BUCKET_NAME  # project bucket — raw + preprocessed images live here
model_package_group_name = "PneumoniaCNNModelPackageGroup"

print(f"Region:                {region}")
print(f"Role:                  {role}")
print(f"Default bucket:        {default_bucket}")
print(f"Project bucket:        {bucket}")
print(f"Model package group:   {model_package_group_name}")

## Step 2 · Pipeline Parameters

Parameters become the tunable knobs on every execution. The same registered pipeline can be re-run via `pipeline.start(parameters={...})` with different values — no notebook edits, no code review.

| Parameter | Production knob |
|---|---|
| `ProcessingInstanceType` / `Count` | Scale the split step horizontally if the metadata CSV grows. |
| `TrainingInstanceType` | CPU (`ml.m5.xlarge`) for safety; flip to GPU (`ml.g4dn.xlarge`) for faster training. |
| `TrainingEpochs`, `TrainingBatchSize`, `LearningRate` | Tune without code changes. |
| `ModelApprovalStatus` | `PendingManualApproval` → human flips in Model Registry → deployment automation picks up. |
| `InputMetadataS3Uri`, `BatchInferenceS3Uri` | Point at a different dataset for a retrain or a different batch slice. |
| `F1Threshold` | Production quality gate — raise to tighten, lower to ship more permissively. |


In [ ]:
from sagemaker.workflow.parameters import (
    ParameterInteger,
    ParameterString,
    ParameterFloat,
)

# Compute defaults from the project bucket so the pipeline runs against the same
# artifacts the rest of the project does. Override at execution time as needed.
default_metadata_uri = f"s3://{bucket}/pneumonia-project/metadata/image_metadata.csv"
default_batch_uri = f"s3://{bucket}/preprocessed-images/"

# --- Infrastructure -----------------------------------------------------
processing_instance_type = ParameterString(
    name="ProcessingInstanceType", default_value="ml.m5.xlarge"
)
processing_instance_count = ParameterInteger(
    name="ProcessingInstanceCount", default_value=1
)
training_instance_type = ParameterString(
    name="TrainingInstanceType", default_value="ml.m5.xlarge"
)
training_instance_count = ParameterInteger(
    name="TrainingInstanceCount", default_value=1
)

# --- Training hyperparameters ------------------------------------------
training_epochs = ParameterInteger(name="TrainingEpochs", default_value=10)
training_batch_size = ParameterInteger(name="TrainingBatchSize", default_value=16)
learning_rate = ParameterFloat(name="LearningRate", default_value=1e-3)

# --- Data + deployment --------------------------------------------------
input_metadata_uri = ParameterString(
    name="InputMetadataS3Uri", default_value=default_metadata_uri
)
batch_inference_uri = ParameterString(
    name="BatchInferenceS3Uri", default_value=default_batch_uri
)
model_approval_status = ParameterString(
    name="ModelApprovalStatus", default_value="PendingManualApproval"
)

# --- Quality gate -------------------------------------------------------
# Higher is better — opposite of the abalone-MSE lab example.
f1_threshold = ParameterFloat(name="F1Threshold", default_value=0.85)

## Step 3 · ProcessingStep — Split & Materialize Manifests

Read the master metadata CSV (which already points to preprocessed PNGs on S3), do the canonical stratified 40/10/10/40 split, and emit four manifest CSVs the training and evaluation scripts will consume.

We compute `class_weight` here as well (Normal ~0.73, Pneumonia ~1.58 for a ~2:1 imbalance) and pickle it into the train channel so the training script doesn't need access to the full manifest.

Why a Processing step for what's basically `pandas.train_test_split`? Two reasons: (1) SageMaker tracks step lineage — every downstream artifact is linked back to this split, and (2) the step is parameterized, so the split logic is part of the registered pipeline rather than living in notebook state.


In [ ]:
!mkdir -p pipeline_scripts

In [ ]:
%%writefile pipeline_scripts/preprocessing.py
"""Stratified 40/10/10/40 split of the project's preprocessed-image manifest.

Reads:  /opt/ml/processing/input/image_metadata.csv
Writes: /opt/ml/processing/train/manifest.csv          (~40% of rows)
        /opt/ml/processing/train/class_weights.json    (class -> float)
        /opt/ml/processing/validation/manifest.csv     (~10%)
        /opt/ml/processing/test/manifest.csv           (~10%)
        /opt/ml/processing/production/manifest.csv     (~40% — monitoring slice)

All four manifests share the schema of the input CSV, so the training and
evaluation scripts only need to know how to read one shape.
"""
import argparse
import json
import pathlib

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight


def main(args):
    base = pathlib.Path("/opt/ml/processing")
    df = pd.read_csv(base / "input" / "image_metadata.csv")

    # The data_preparations notebook emits manifests with a `preprocessed_s3_key`
    # column when preprocessing has run. If we're somehow handed the pre-preprocess
    # schema (only `s3_key`), fail loudly with a clear error.
    if "preprocessed_s3_key" not in df.columns:
        raise ValueError(
            "Input manifest is missing 'preprocessed_s3_key'. Run "
            "data_preparations.ipynb with IS_DATA_OWNER=True first."
        )

    df = df.dropna(subset=["label", "label_int", "preprocessed_s3_key"]).reset_index(drop=True)
    print(f"Input rows: {len(df)}")
    print(f"Class counts: {df['label'].value_counts().to_dict()}")

    # Three-step split keeps the production slice cleanly separated from the
    # train/val/test triple. Stratify on label to preserve the ~2:1 ratio.
    df_model, df_prod = train_test_split(
        df, test_size=0.40, random_state=args.random_state, stratify=df["label"]
    )
    df_train, df_temp = train_test_split(
        df_model, test_size=0.333, random_state=args.random_state,
        stratify=df_model["label"],
    )
    df_test, df_val = train_test_split(
        df_temp, test_size=0.5, random_state=args.random_state,
        stratify=df_temp["label"],
    )

    # Compute class weights on the train slice (matches CNN_Model.ipynb).
    classes = sorted(df_train["label_int"].unique())
    weights = compute_class_weight("balanced", classes=classes, y=df_train["label_int"])
    class_weight = {int(c): float(w) for c, w in zip(classes, weights)}
    print(f"class_weight: {class_weight}")

    for name, frame in [
        ("train", df_train),
        ("validation", df_val),
        ("test", df_test),
        ("production", df_prod),
    ]:
        out_dir = base / name
        out_dir.mkdir(parents=True, exist_ok=True)
        frame.to_csv(out_dir / "manifest.csv", index=False)
        print(f"{name:11s} -> {len(frame):6d} rows")

    with open(base / "train" / "class_weights.json", "w") as f:
        json.dump(class_weight, f)


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--random-state", type=int, default=42)
    main(parser.parse_args())

In [ ]:
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.workflow.steps import ProcessingStep

split_processor = SKLearnProcessor(
    framework_version="1.2-1",
    instance_type=processing_instance_type,
    instance_count=processing_instance_count,
    base_job_name="pneumonia-split",
    role=role,
    sagemaker_session=pipeline_session,
)

# pipeline_session captures these args instead of running the job.
split_args = split_processor.run(
    inputs=[
        ProcessingInput(
            source=input_metadata_uri,
            destination="/opt/ml/processing/input",
        ),
    ],
    outputs=[
        ProcessingOutput(output_name="train",      source="/opt/ml/processing/train"),
        ProcessingOutput(output_name="validation", source="/opt/ml/processing/validation"),
        ProcessingOutput(output_name="test",       source="/opt/ml/processing/test"),
        ProcessingOutput(output_name="production", source="/opt/ml/processing/production"),
    ],
    code="pipeline_scripts/preprocessing.py",
)

step_split = ProcessingStep(name="PneumoniaSplit", step_args=split_args)

## Step 4 · TrainingStep — Pneumonia CNN

Wraps the same 4-block CNN from `CNN_Model.ipynb` as a SageMaker TensorFlow training job. The training script reads the train/validation manifests from the two channels the previous step produces, streams PNGs from S3 into `tf.data.Dataset` pipelines, trains the CNN with class weights + augmentation + early stopping, and emits both the Keras checkpoint and an ONNX export.

Estimator output:
- `/opt/ml/model/pneumonia_cnn_model.keras` — best checkpoint (used by evaluation step).
- `/opt/ml/model/pneumonia_cnn_model.onnx` — framework-agnostic export (used by the deploy story in `Model_Monitoring.ipynb`).
- `/opt/ml/output/data/training_history.json` — loss/accuracy curves for post-mortem.


In [ ]:
%%writefile pipeline_scripts/train.py
"""Train the pneumonia CNN inside SageMaker's managed TensorFlow container.

Channels (injected by SageMaker):
  /opt/ml/input/data/train/manifest.csv
  /opt/ml/input/data/train/class_weights.json
  /opt/ml/input/data/validation/manifest.csv

Outputs:
  /opt/ml/model/pneumonia_cnn_model.keras
  /opt/ml/model/pneumonia_cnn_model.onnx
  /opt/ml/output/data/training_history.json
"""
import argparse
import json
import os
import pathlib
from urllib.parse import urlparse

import boto3
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

IMG_SIZE = 128


def _parse_s3(uri):
    p = urlparse(uri)
    return p.netloc, p.path.lstrip("/")


_s3 = boto3.client("s3")


def _load_png_from_s3(s3_uri):
    bucket, key = _parse_s3(s3_uri.numpy().decode("utf-8"))
    obj = _s3.get_object(Bucket=bucket, Key=key)
    return obj["Body"].read()


def _decode_image(raw_bytes):
    img = tf.io.decode_png(raw_bytes, channels=1)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    img = tf.cast(img, tf.float32) / 255.0
    return img


def _build_dataset(manifest_path, batch_size, augment):
    df = pd.read_csv(manifest_path)
    keys = df["preprocessed_s3_key"].tolist()
    labels = df["label_int"].astype(np.float32).tolist()

    def _map(uri, label):
        raw = tf.py_function(_load_png_from_s3, [uri], tf.string)
        img = _decode_image(raw)
        img.set_shape([IMG_SIZE, IMG_SIZE, 1])
        return img, label

    ds = tf.data.Dataset.from_tensor_slices((keys, labels))
    ds = ds.map(_map, num_parallel_calls=tf.data.AUTOTUNE)
    if augment:
        aug = tf.keras.Sequential([
            layers.RandomFlip("horizontal"),
            layers.RandomRotation(0.05),
            layers.RandomZoom(0.1),
            layers.RandomContrast(0.1),
            layers.RandomTranslation(0.05, 0.05),
        ])
        ds = ds.map(lambda x, y: (aug(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
        ds = ds.shuffle(2048)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)


def _build_model():
    # 4-block VGG-style CNN; mirrors CNN_Model.ipynb.
    def block(x, filters):
        x = layers.Conv2D(filters, 3, padding="same")(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
        x = layers.Conv2D(filters, 3, padding="same")(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
        return layers.MaxPooling2D()(x)

    inp = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 1))
    x = block(inp, 32)
    x = block(x, 64)
    x = block(x, 128)
    x = block(x, 256)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    return models.Model(inp, out)


def main(args):
    train_manifest = pathlib.Path(args.train_dir) / "manifest.csv"
    val_manifest   = pathlib.Path(args.val_dir) / "manifest.csv"
    weights_path   = pathlib.Path(args.train_dir) / "class_weights.json"

    with open(weights_path) as f:
        class_weight = {int(k): float(v) for k, v in json.load(f).items()}
    print(f"class_weight: {class_weight}")

    train_ds = _build_dataset(train_manifest, args.batch_size, augment=True)
    val_ds   = _build_dataset(val_manifest,   args.batch_size, augment=False)

    model = _build_model()
    model.compile(
        optimizer=tf.keras.optimizers.Adam(args.learning_rate),
        loss="binary_crossentropy",
        metrics=["accuracy", tf.keras.metrics.AUC(name="auc")],
    )

    cb = [
        callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3),
    ]
    history = model.fit(
        train_ds, validation_data=val_ds,
        epochs=args.epochs, class_weight=class_weight,
        callbacks=cb,
    )

    out_model_dir = pathlib.Path(args.model_dir)
    out_model_dir.mkdir(parents=True, exist_ok=True)
    model.save(out_model_dir / "pneumonia_cnn_model.keras")

    # ONNX export — same artifact Model_Monitoring.ipynb uses to deploy.
    try:
        import tf2onnx
        spec = (tf.TensorSpec((None, IMG_SIZE, IMG_SIZE, 1), tf.float32, name="input"),)
        tf2onnx.convert.from_keras(
            model, input_signature=spec, opset=13,
            output_path=str(out_model_dir / "pneumonia_cnn_model.onnx"),
        )
        print("ONNX export OK")
    except Exception as e:
        print(f"ONNX export skipped: {type(e).__name__}: {e}")

    history_dict = {k: [float(v) for v in vals] for k, vals in history.history.items()}
    out_data = pathlib.Path(args.output_data_dir)
    out_data.mkdir(parents=True, exist_ok=True)
    with open(out_data / "training_history.json", "w") as f:
        json.dump(history_dict, f, indent=2)


if __name__ == "__main__":
    p = argparse.ArgumentParser()
    p.add_argument("--epochs",        type=int,   default=10)
    p.add_argument("--batch-size",    type=int,   default=16)
    p.add_argument("--learning-rate", type=float, default=1e-3)
    p.add_argument("--train-dir",       default=os.environ["SM_CHANNEL_TRAIN"])
    p.add_argument("--val-dir",         default=os.environ["SM_CHANNEL_VALIDATION"])
    p.add_argument("--model-dir",       default=os.environ.get("SM_MODEL_DIR", "/opt/ml/model"))
    p.add_argument("--output-data-dir", default=os.environ.get("SM_OUTPUT_DATA_DIR", "/opt/ml/output/data"))
    # SageMaker TF estimator always injects --model_dir (underscore) for
    # distributed-training plumbing. We don't use it but argparse needs it.
    p.add_argument("--model_dir", default=None)
    main(p.parse_args())

In [ ]:
from sagemaker.tensorflow import TensorFlow
from sagemaker.inputs import TrainingInput
from sagemaker.workflow.steps import TrainingStep

# TF 2.13 is the closest pinned version SageMaker offers to the project's local
# 2.19 training environment. Behavior of the layers we use is stable across
# both — no API changes between minor versions for our layer set.
cnn_estimator = TensorFlow(
    entry_point="train.py",
    source_dir="pipeline_scripts",
    framework_version="2.19",
    py_version="py312",
    instance_type=training_instance_type,
    instance_count=training_instance_count,
    role=role,
    sagemaker_session=pipeline_session,
    output_path=f"s3://{default_bucket}/PneumoniaCNNTrain",
    hyperparameters={
        "epochs":        training_epochs,
        "batch-size":    training_batch_size,
        "learning-rate": learning_rate,
    },
    # tf2onnx isn't in the base TF training image; install it on container start.
    # SageMaker honors requirements.txt next to entry_point automatically.
)

train_args = cnn_estimator.fit(
    inputs={
        "train": TrainingInput(
            s3_data=step_split.properties.ProcessingOutputConfig.Outputs["train"].S3Output.S3Uri,
            content_type="text/csv",
        ),
        "validation": TrainingInput(
            s3_data=step_split.properties.ProcessingOutputConfig.Outputs["validation"].S3Output.S3Uri,
            content_type="text/csv",
        ),
    }
)

step_train = TrainingStep(name="PneumoniaTrain", step_args=train_args)

### 4.1 Pipeline requirements for the training container

SageMaker's TensorFlow training container doesn't ship `tf2onnx`. Drop a `requirements.txt` next to the training script so the container installs it before invoking `train.py`.

In [ ]:
%%writefile pipeline_scripts/requirements.txt
tf2onnx>=1.16
onnx>=1.14


## Step 5 · ProcessingStep — Evaluate the Trained Model

Run the trained model against the test manifest, compute the full medical-AI metric suite, and write `evaluation.json` in the SageMaker Model Quality format. The next step's `ConditionStep` reads `binary_classification_metrics.f1.value` out of this JSON via `JsonGet`.

**Threshold = 0.4** here mirrors the design doc's clinical-stakes choice (favors recall — missing pneumonia is the costly error).

In [ ]:
%%writefile pipeline_scripts/evaluation.py
"""Evaluate the trained CNN against the test manifest.

Inputs:  /opt/ml/processing/model/model.tar.gz
         /opt/ml/processing/test/manifest.csv
Output:  /opt/ml/processing/evaluation/evaluation.json
"""
import json
import pathlib
import tarfile
from urllib.parse import urlparse

import boto3
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import (
    confusion_matrix, precision_score, recall_score, f1_score,
    accuracy_score, roc_auc_score,
)

IMG_SIZE = 128
THRESHOLD = 0.4  # favors recall — see knowledge/05-model-and-training.md

_s3 = boto3.client("s3")


def _parse_s3(uri):
    p = urlparse(uri)
    return p.netloc, p.path.lstrip("/")


def _load_image(s3_uri):
    b, k = _parse_s3(s3_uri)
    raw = _s3.get_object(Bucket=b, Key=k)["Body"].read()
    img = tf.io.decode_png(raw, channels=1)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    return (tf.cast(img, tf.float32) / 255.0).numpy()


def main():
    model_tar = pathlib.Path("/opt/ml/processing/model/model.tar.gz")
    extract_to = pathlib.Path("/tmp/model")
    extract_to.mkdir(parents=True, exist_ok=True)
    with tarfile.open(model_tar) as t:
        t.extractall(extract_to)

    # The TensorFlow estimator saves the Keras file at the model dir root.
    keras_path = next(extract_to.rglob("pneumonia_cnn_model.keras"), None)
    if keras_path is None:
        # Fallback for older SavedModel layout.
        sm_dir = next(p.parent for p in extract_to.rglob("saved_model.pb"))
        model = tf.keras.models.load_model(sm_dir)
    else:
        model = tf.keras.models.load_model(keras_path)

    df = pd.read_csv("/opt/ml/processing/test/manifest.csv")
    print(f"Evaluating on {len(df)} test images at threshold={THRESHOLD}")

    # Load + predict in batches to keep memory bounded.
    BATCH = 64
    probs = []
    for start in range(0, len(df), BATCH):
        chunk = df.iloc[start:start + BATCH]
        imgs = np.stack([_load_image(uri) for uri in chunk["preprocessed_s3_key"]])
        probs.append(model.predict(imgs, verbose=0).reshape(-1))
    y_prob = np.concatenate(probs)
    y_pred = (y_prob >= THRESHOLD).astype(int)
    y_true = df["label_int"].to_numpy().astype(int)

    cm = confusion_matrix(y_true, y_pred).tolist()  # [[TN, FP], [FN, TP]]
    report = {
        "binary_classification_metrics": {
            "accuracy":  {"value": float(accuracy_score(y_true, y_pred)),  "standard_deviation": "NaN"},
            "precision": {"value": float(precision_score(y_true, y_pred)), "standard_deviation": "NaN"},
            "recall":    {"value": float(recall_score(y_true, y_pred)),    "standard_deviation": "NaN"},
            "f1":        {"value": float(f1_score(y_true, y_pred)),        "standard_deviation": "NaN"},
            "auc":       {"value": float(roc_auc_score(y_true, y_prob)),   "standard_deviation": "NaN"},
            "confusion_matrix": cm,
            "threshold": THRESHOLD,
        }
    }
    print(json.dumps(report, indent=2))

    out = pathlib.Path("/opt/ml/processing/evaluation")
    out.mkdir(parents=True, exist_ok=True)
    with open(out / "evaluation.json", "w") as f:
        json.dump(report, f)


if __name__ == "__main__":
    main()

In [ ]:
from sagemaker.processing import ScriptProcessor
from sagemaker.workflow.properties import PropertyFile

# Reuse the TensorFlow inference image so the eval script has `tensorflow`
# available without us building a custom container.
tf_image_uri = sagemaker.image_uris.retrieve(
    framework="tensorflow",
    region=region,
    version="2.19",
    py_version="py312",
    instance_type="ml.m5.xlarge",
    image_scope="training",
)

eval_processor = ScriptProcessor(
    image_uri=tf_image_uri,
    command=["python3"],
    instance_type="ml.m5.xlarge",
    instance_count=1,
    base_job_name="pneumonia-eval",
    role=role,
    sagemaker_session=pipeline_session,
)

eval_args = eval_processor.run(
    inputs=[
        ProcessingInput(
            source=step_train.properties.ModelArtifacts.S3ModelArtifacts,
            destination="/opt/ml/processing/model",
        ),
        ProcessingInput(
            source=step_split.properties.ProcessingOutputConfig.Outputs["test"].S3Output.S3Uri,
            destination="/opt/ml/processing/test",
        ),
    ],
    outputs=[
        ProcessingOutput(output_name="evaluation", source="/opt/ml/processing/evaluation"),
    ],
    code="pipeline_scripts/evaluation.py",
)

evaluation_report = PropertyFile(
    name="EvaluationReport",
    output_name="evaluation",
    path="evaluation.json",
)

step_eval = ProcessingStep(
    name="PneumoniaEvaluate",
    step_args=eval_args,
    property_files=[evaluation_report],
)

## Step 6 · Conditional Branch on Model Quality

If the trained model clears the F1 bar:
1. **Register** in the SageMaker Model Registry with `ModelApprovalStatus` (default: `PendingManualApproval`) so a human gates production deploy.
2. **Create** a `Model` resource so the batch-transform step has something to invoke.
3. **BatchTransform** over the production-slice manifest the split step produced (the same data the monitoring pipeline will consume).

If the model misses the bar:
- **FailStep** ends the execution with a human-readable error containing the actual F1 (so it's visible in the SageMaker Studio Pipelines UI without digging into S3).

In [ ]:
from sagemaker.model import Model
from sagemaker.workflow.model_step import ModelStep
from sagemaker.model_metrics import MetricsSource, ModelMetrics

# Inference image: reuse the same TF image (it serves Keras SavedModel out of the box).
inference_image_uri = sagemaker.image_uris.retrieve(
    framework="tensorflow",
    region=region,
    version="2.19",
    py_version="py312",
    instance_type="ml.m5.large",
    image_scope="inference",
)

model = Model(
    image_uri=inference_image_uri,
    model_data=step_train.properties.ModelArtifacts.S3ModelArtifacts,
    sagemaker_session=pipeline_session,
    role=role,
)

model_metrics = ModelMetrics(
    model_statistics=MetricsSource(
        s3_uri="{}/evaluation.json".format(
            step_eval.arguments["ProcessingOutputConfig"]["Outputs"][0]["S3Output"]["S3Uri"]
        ),
        content_type="application/json",
    )
)

register_args = model.register(
    content_types=["application/x-image"],
    response_types=["application/json"],
    inference_instances=["ml.m5.large", "ml.m5.xlarge"],
    transform_instances=["ml.m5.xlarge"],
    model_package_group_name=model_package_group_name,
    approval_status=model_approval_status,
    model_metrics=model_metrics,
)
step_register = ModelStep(name="PneumoniaRegister", step_args=register_args)

step_create_model = ModelStep(
    name="PneumoniaCreateModel",
    step_args=model.create(instance_type="ml.m5.large"),
)

In [ ]:
from sagemaker.transformer import Transformer
from sagemaker.inputs import TransformInput
from sagemaker.workflow.steps import TransformStep

transformer = Transformer(
    model_name=step_create_model.properties.ModelName,
    instance_type="ml.m5.xlarge",
    instance_count=1,
    output_path=f"s3://{default_bucket}/PneumoniaBatchTransform",
    sagemaker_session=pipeline_session,
)

step_transform = TransformStep(
    name="PneumoniaBatchTransform",
    transformer=transformer,
    inputs=TransformInput(data=batch_inference_uri),
)

In [ ]:
from sagemaker.workflow.fail_step import FailStep
from sagemaker.workflow.functions import Join, JsonGet

# Embed the actual F1 value into the failure message so the SageMaker Pipelines
# UI shows *why* it failed without needing to open evaluation.json.
failed_f1 = JsonGet(
    step_name=step_eval.name,
    property_file=evaluation_report,
    json_path="binary_classification_metrics.f1.value",
)

step_fail = FailStep(
    name="PneumoniaF1Fail",
    error_message=Join(
        on=" ",
        values=["Execution failed: F1", failed_f1, "<", f1_threshold],
    ),
)

In [ ]:
from sagemaker.workflow.conditions import ConditionGreaterThanOrEqualTo
from sagemaker.workflow.condition_step import ConditionStep

cond_f1 = ConditionGreaterThanOrEqualTo(
    left=JsonGet(
        step_name=step_eval.name,
        property_file=evaluation_report,
        json_path="binary_classification_metrics.f1.value",
    ),
    right=f1_threshold,
)

step_cond = ConditionStep(
    name="PneumoniaQualityGate",
    conditions=[cond_f1],
    if_steps=[step_register, step_create_model, step_transform],
    else_steps=[step_fail],
)

## Step 7 · Assemble the Pipeline

`steps=[step_split, step_train, step_eval, step_cond]` — the ConditionStep recursively includes its `if_steps` and `else_steps`, so we don't list the register/create/transform/fail steps at the top level. SageMaker resolves the dependency graph from the property references between steps.

In [ ]:
from sagemaker.workflow.pipeline import Pipeline

pipeline_name = "PneumoniaCNNPipeline"
pipeline = Pipeline(
    name=pipeline_name,
    parameters=[
        processing_instance_type,
        processing_instance_count,
        training_instance_type,
        training_instance_count,
        training_epochs,
        training_batch_size,
        learning_rate,
        input_metadata_uri,
        batch_inference_uri,
        model_approval_status,
        f1_threshold,
    ],
    steps=[step_split, step_train, step_eval, step_cond],
    sagemaker_session=pipeline_session,
)

In [ ]:
# Sanity-check: pipeline definition should be a well-formed dict with all
# 9 step names (Split, Train, Evaluate, the Condition step, and the four
# inner steps in its branches: Register, CreateModel, BatchTransform, Fail).
definition = json.loads(pipeline.definition())
step_names = [s["Name"] for s in definition["Steps"]]
print("Top-level steps:", step_names)
print("Parameters:    ", [p["Name"] for p in definition["Parameters"]])

## Step 8 · Register and Run

`upsert` creates the pipeline (or updates it in place if a `PneumoniaCNNPipeline` already exists in this account/region). After this call the DAG is visible in the SageMaker Studio Pipelines view and can be triggered from outside the notebook (CLI, EventBridge, GitHub Actions, etc.).

`start()` triggers an execution with default parameter values. The notebook's job ends at the start call — SageMaker drives every step from there.


In [ ]:
pipeline.upsert(role_arn=role)
print(f"Pipeline upserted: {pipeline_name}")

In [ ]:
execution = pipeline.start()
execution.describe()

In [ ]:
# Blocks the notebook until SageMaker reports the pipeline as completed,
# failed, or stopped. Typical end-to-end time on the default parameters:
#   ~3 min split + ~30-60 min train (CPU) + ~5 min eval + ~5 min batch transform.
execution.wait()
execution.list_steps()

## Step 9 · Inspect the Evaluation Report

Pull the `evaluation.json` the evaluation step produced and confirm which branch the ConditionStep took. If F1 >= the threshold (default 0.85), the Register/Create/BatchTransform steps will be `Succeeded` in `list_steps` above; if not, you'll see `PneumoniaF1Fail` instead.

In [ ]:
from pprint import pprint

eval_output_uri = step_eval.arguments["ProcessingOutputConfig"]["Outputs"][0]["S3Output"]["S3Uri"]
evaluation_json = sagemaker.s3.S3Downloader.read_file(
    f"{eval_output_uri}/evaluation.json",
    sagemaker_session=sagemaker_session,
)
report = json.loads(evaluation_json)
pprint(report)

metrics = report["binary_classification_metrics"]
print(f"\nF1: {metrics['f1']['value']:.4f}")
print(f"Threshold (gate): {f1_threshold.default_value}")
print(f"Verdict: {'PASS' if metrics['f1']['value'] >= f1_threshold.default_value else 'FAIL'}")

## Step 10 · Parametrized Re-runs (CI/CD Demonstration)

Same registered pipeline, different parameter values. No notebook edit, no code review, no new artifact. This is the value of the pipeline-as-DAG pattern — operations can tune knobs without engineering involvement.


### 10.1 Tighter F1 gate → forces the FailStep

Raising `F1Threshold` to 0.99 guarantees the trained model misses the bar (no real model hits 0.99 on this dataset). Demonstrates that the ConditionStep is wired correctly: `PneumoniaF1Fail` runs and the execution is marked failed, with the actual F1 in the error message.

In [ ]:
execution_tight = pipeline.start(parameters=dict(F1Threshold=0.99))
try:
    execution_tight.wait()
except Exception as e:
    print(f"Expected failure: {e}")
execution_tight.list_steps()

### 10.2 Auto-approve model for deploy

Override `ModelApprovalStatus` to `Approved` so downstream deploy automation can pick this model package up without a human in the loop. (Used for the green-light branch in CI — main-branch merges register `PendingManualApproval`; releases tagged `*-prod` register `Approved`.)

In [ ]:
execution_approved = pipeline.start(parameters=dict(ModelApprovalStatus="Approved"))
execution_approved.wait()
execution_approved.list_steps()

## Step 11 · Lineage Visualization

SageMaker tracks lineage automatically — every artifact (raw images, preprocessed PNGs, train/val/test manifests, training job, model artifact, metrics, batch transform output) is linked through the execution graph. The visualizer walks the graph and renders it as a table.


In [ ]:
import time
from sagemaker.lineage.visualizer import LineageTableVisualizer

viz = LineageTableVisualizer(sagemaker_session)
# Walk the latest run's steps in reverse order (artifacts flow downstream).
for step in reversed(execution.list_steps()):
    print(f"\n== {step['StepName']} ==")
    display(viz.show(pipeline_execution_step=step))
    time.sleep(2)  # avoid throttling on the Lineage API

## Step 12 · CI/CD Hooks Around the Pipeline

The SageMaker Pipeline is the *execution* half of CI/CD. The *trigger* half lives in GitHub Actions and the *deploy* half lives downstream of the Model Registry. Sketch of the full lifecycle:

```
Developer push  ─────────────────────────────────────────────────►
                                                                  │
                                                                  ▼
              GitHub Actions (.github/workflows/test.yml)
                ├─ Lint + unit tests on img_preprocessing.py
                ├─ Schema test on image_metadata.csv
                ├─ Stratification / no-leakage test on the split step
                ├─ Smoke train (1 epoch on a tiny subset)
                └─ tf2onnx round-trip test
                          │
                          ▼  (on PR merge to main)
              GitHub Actions → boto3 → pipeline.start()
                          ModelApprovalStatus=PendingManualApproval
                          │
                          ▼
              SageMaker Pipeline (THIS NOTEBOOK)
                ├─ Split  ─►  Train  ─►  Evaluate  ─►  ConditionStep
                                                          │
                                  ┌───────────────────────┴────────────┐
                                  ▼                                    ▼
                       Register + CreateModel               FailStep
                          + BatchTransform                  (PR blocked from
                                  │                          merging to prod)
                                  ▼
              Model Registry (status=PendingManualApproval)
                          │
                          ▼  (human flips to Approved)
              EventBridge rule → Lambda → endpoint update
                          │
                          ▼
              Live SageMaker Endpoint (Model_Monitoring.ipynb watches)
```

**What this notebook delivers vs the broader CI/CD picture:**

| Piece | Where it lives | Status |
|---|---|---|
| Pipeline definition (DAG) | this notebook | ✅ implemented |
| Parameter-driven re-runs | this notebook | ✅ demonstrated in §10 |
| Model quality gate (F1 ≥ threshold) | this notebook | ✅ wired via ConditionStep |
| Model Registry hand-off | this notebook | ✅ via ModelStep(register) |
| Lineage tracking | SageMaker built-in | ✅ shown in §11 |
| GitHub Actions trigger | `.github/workflows/` | ⏳ scoped in `knowledge/08-cicd.md`; not yet in repo |
| Unit / schema / leakage tests | `tests/` | ⏳ scoped in design doc; not yet in repo |
| Endpoint deploy on approval | EventBridge + Lambda | ⏳ non-goal per design doc (`07-deployment-monitoring.md`) |

The right column items are explicit follow-ups, scoped in the project knowledge base but out of scope for the course deliverable's primary CI/CD notebook.